In [2]:
# 加载环境变量
from dotenv import load_dotenv

load_dotenv()

True

## 1.初始化并调用模型

langchain提供了两种常见方法用来初始化模型：
- 使用init_chat_model方法，由langchain自动创建模型对象
- 使用不同模型对应的Model类，手动创建模型对象


### 1.1.init_chat_model
使用init_chat_model方法，langchain根据模型名称自动初始化与模型的连接，非常方便。

但需要注意的是：**如果使用我们必须在.env中配置好模型提供者的api_key**


In [ ]:
# 导入Langchain的初始化模型的函数
from langchain.chat_models import init_chat_model

# 调用init_chat_model函数初始化模型，参数model用来指定模型名称，Langchain会根据模型名字自动设定base_url，并从环境变量中获取api_key
model = init_chat_model(model="deepseek-chat")

In [3]:
# init_chat_model返回的模型会根据模型名称自动确定其类型
print(type(model))

<class 'langchain_deepseek.chat_models.ChatDeepSeek'>


## 1.2.自定义模型

init_chat_model默认会根据模型名称自动确定模型的提供者、其base_url，并从env读取api_key，但前提是必须是langchain支持的模型平台，例如：
- openai
- deepseek
- ...

对于其它模型，我们必须自定义模型参数来访问。

例如，我们要访问阿里云百炼的qwen-max，它就是不被langchain支持的模型，我们必须自定义模型参数来访问。
- 我们需要在环境变量中定义api_key和base_url
- 然后在init_chat_model中指定model、model_provider、base_url和api_key


In [37]:
# 我们收到加载环境变量中的base_url和api_key
import os

base_url = os.getenv("DASHSCOPE_BASE_URL")
api_key = os.getenv("DASHSCOPE_API_KEY")

model = init_chat_model(
    model="qwen-max",  # 模型名称，这里可以自定义，我们用的是阿里的qwen-max
    model_provider="openai",  # 如果是Langchain不支持的模型，需要指定模型提供者（虽然我们用的是阿里，但是阿里兼容openai，所以这里用openai）
    base_url=base_url,
    api_key=api_key
)

In [38]:
# 自定义模型参数时，模型的类型由model_provider确定
print(type(model))

<class 'langchain_openai.chat_models.base.ChatOpenAI'>


## 1.3.调整模型参数
除了修改模型提供者以外，init_chat_model方法允许我们调整模型参数，例如：
- temperature: 控制生成文本的随机性，值越小越确定，值越大越随机
- max_tokens: 控制生成文本的最大长度
- top_p: 控制生成文本的多样性，值越小越多样，值越大越确定
- timeout: 控制生成文本的超时时间
- max_retries: 控制生成文本的最大重试次数
- ...


In [39]:
# 调用init_chat_model函数初始化模型，并设定模型参数
model = init_chat_model(
    model="qwen-max",  # 模型名称，这里可以自定义，我们用的是阿里的qwen-max
    model_provider="openai",  # 如果是Langchain不支持的模型，需要指定模型提供者（虽然我们用的是阿里，但是阿里兼容openai，所以这里用openai）
    base_url=base_url,
    api_key=api_key,
    temperature=1.5,
)

# 自定义模型参数时，模型的类型由model_provider确定
print(type(model))


<class 'langchain_openai.chat_models.base.ChatOpenAI'>


## 1.4.使用model类
其实init_chat_model方法底层就是帮我们利用Model类创建对象。但只支持有限的模型。

而在langchain的社区，除了langchain官方提供的Model，还有些类是社区提供，更丰富多样。

具体支持的模型，可以查看官网地址：https://docs.langchain.com/oss/python/integrations/chat



例如，我们使用社区版本的Model类来访问阿里云百炼的通义千问模型：

1. 首先，我们需要安装依赖
    LangChain社区依赖：
    ```bash
    uv add langchain-community
    ```
    阿里云百炼依赖：
    ```bash
   uv add dashscope
   ```
2. 然后，我们就可以使用Model类初始化模型了


In [3]:
from langchain_community.chat_models.tongyi import ChatTongyi

# 使用Model类初始化模型
model = ChatTongyi(
    model="qwen-max"
    # 其它模型参数...
)

In [4]:
# 打印结果
print(type(model))

<class 'langchain_community.chat_models.tongyi.ChatTongyi'>


# 2.访问模型

LangChain提供了两个不同的方法来访问模型：
- invoke：阻塞式访问
- stream：流式访问

## 2.1.invoke
invoke方法是阻塞式调用，需要等待模型生成全部结果才会返回，等待时间较长。


In [58]:
# 通过invoke方法访问模型，需要阻塞等待模型生成结果
response = model.invoke("月亮的首都是哪里？")

In [59]:
# 查看响应内容
print(response)

content='月亮并没有首都，因为它不是国家或政治实体，而是一个天体。首都是指一个国家的政府所在地或行政中心，比如中国的首都是北京，美国的首都是华盛顿特区。月球上目前没有常住居民，也没有设立任何形式的政府机构。不过，人类对于月球探索的兴趣一直很浓厚，未来也许会有更多的探索和发展计划。' additional_kwargs={} response_metadata={'model_name': 'qwen-max', 'finish_reason': 'stop', 'request_id': '3d4670a4-db02-49d1-a1ed-7538ce228f50', 'token_usage': {'input_tokens': 14, 'output_tokens': 78, 'prompt_tokens_details': {'cached_tokens': 0}, 'total_tokens': 92}} id='lc_run--019c9e7c-955e-7110-96d6-ac3d26d530f8-0' tool_calls=[] invalid_tool_calls=[]


## 2.2.流式访问

阻塞式调用需要等待较长时间才能看到AI返回的结果，而流式调用则可以实时看到AI返回的一个个词。

In [5]:
# 通过.stream方法实现流式访问
stream = model.stream("月亮的首都是哪里？")

# 流式访问会返回一个生成器对象，可以遍历生成器对象来实时获取AI的回复
print(type(stream))

<class 'generator'>


In [6]:
# 遍历stream结果，实时打印AI的回复
for chunk in stream:
    print(chunk.content, end="", flush=True)

月亮是地球的自然卫星，并不是一个国家或行政区域，因此没有首都。这个问题可能是在以一种幽默或比喻的方式提出。如果你有关于月亮的具体问题，比如它的地理特征、探索历史等，我很乐意为你解答！

# 3.在智能体中使用模型

本节我们学习如何在智能体中使用模型。

## 3.1.创建智能体
Langchain提供了一个create_agent方法用来快速创建智能体。调用create_agent时需要指定一个模型。有两种选择：
- 使用初始化好的模型对象
- 使用模型名称，让Langchain自动初始化模型


In [9]:
from langchain.agents import create_agent

# 1.使用初始化好的model创建Agent
agent = create_agent(model=model)

In [10]:
# 2.指定Model名称，由LangChain自动初始化模型
agent = create_agent(model="deepseek-chat")

## 3.2.调用智能体

智能体调用与模型调用类似，也支持两种方式：
- invoke：阻塞式调用
- stream：流式访问

但需要注意的是，智能体调用时需要传入一个dict，其中必须包含一个messages字段，也就是消息的列表。

### 阻塞式调用

In [11]:
response = agent.invoke({
    "messages": [{"role": "user", "content": "月亮的首都是哪里？"}]
})

print(response)

{'messages': [HumanMessage(content='月亮的首都是哪里？', additional_kwargs={}, response_metadata={}, id='3076a528-6673-475b-a8f9-a389d45e5601'), AIMessage(content='这是一个非常有趣且富有想象力的问题！😊\n\n从科学和现实的角度来说，月亮（月球）上没有城市，也没有首都，因为它是一个没有大气、没有生命、没有国家的自然天体。\n\n不过，从**文学、艺术和文化的角度**来看，这个问题有很多有趣的答案：\n\n1. **科幻作品中的“月球首都”**：\n   - 在很多科幻小说、电影和游戏中，如果人类在月球建立了殖民地，常被称为“月城”、“月球基地”或“月球首都”。\n   - 比如《高达》系列中的“冯·布朗市”，《命运/EXTRA》中的“月之首都”等。\n\n2. **神话传说中的“月亮首都”**：\n   - 在中国神话中，月亮上有“广寒宫”，嫦娥居住在那里。\n   - 在日本传说中，月亮上有“月之都”（月の都），辉夜姬的故事就与此相关。\n\n3. **幽默和创意的回答**：\n   - 有人开玩笑说月亮的首都是“宁静海”（月海名），因为阿波罗11号在那里着陆。\n   - 或者说是“哥白尼环形山”，因为它很显眼。\n\n所以，如果你是在寻找一个富有诗意的答案，可以说**月亮的首都是“广寒宫”**（中国神话），或者**“月之都”**（日本传说）。如果你想要一个科幻版的答案，那就可以自由想象一个名字了！🌙✨\n\n你是对月亮的传说感兴趣，还是在构思什么创意作品呢？', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 312, 'prompt_tokens': 10, 'total_tokens': 322, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_

### 流式访问


In [16]:
# 通过stream方法实现流式访问
messages = agent.stream(
    {"messages": [{"role": "user", "content": "月亮的首都是哪里？"}]},
    stream_mode="messages"
)
print(type( messages))

<class 'generator'>


In [17]:
# 遍历stream结果，实时打印AI的回复
for token, metadata in messages:
    if token.content:  # Check if there's actual content
        print(token.content, end="", flush=True)  # Print token

这是一个非常有趣的问题！从字面上看，“月亮的首都”并不是一个天文学或地理学上的正式概念，因为月亮（月球）上并没有国家或城市。

不过，这个问题可以从几个充满想象力和文化意义的角度来回答：

### 1. **科学/现实角度：人类活动的中心**
如果非要选一个“首都”，那最合理的答案是 **“静海基地”**。
*   这是**阿波罗11号**在1969年首次载人登月的着陆点，是人类在月球上第一个“脚印”所在地，具有无与伦比的历史意义。它象征着人类探索精神的“首都”。

### 2. **科幻/文化角度：虚构的核心**
在科幻作品或流行文化中，有一些著名的虚构月球城市：
*   **“宁静市”**：出自电影《2001太空漫游》，是一个位于月球克莱维斯环形山的美国基地，是故事的关键转折点。
*   **“月球城”或“嫦娥市”**：在许多科幻作品（如《星际牛仔》、《苍穹浩瀚》等）中，月球上往往有一个主要的殖民都市，作为政治和经济中心。
*   在中国文化中，人们可能会联想到 **“广寒宫”** —— 嫦娥居住的月宫，这可以说是月亮上最著名的“宫殿”了。

### 3. **幽默/哲学角度**
*   **“梦”或“想象”**：月亮的“首都”存在于每个人的想象和浪漫情怀中，它是诗歌、艺术和爱情故事的源泉。
*   **“环形山”**：如果月亮是一个国家，那么遍布表面的环形山就是它的主要“城市”和“地貌特征”，其中最大的（如**南极-艾特肯盆地**）或许可以被称为“首都”。

**总结来说：**

如果你想得到一个最具象征意义的答案，那就是 **“静海”**。
如果你想得到一个最富诗意的答案，那就是 **“广寒宫”**。

所以，月亮的“首都”在哪里？它就在**科学、幻想与文化的交汇点上**。